### FlashMTP v1.4 训练启动命令

入口脚本：`scripts/run_training_flashmtp.sh`

- `--dt qz | a800 | h100`：选择机器预设（数据路径 / 目标模型 / WandB 离线等）
- 常用环境变量见下方各 cell；完整列表见 `scripts/run_training_flashmtp.sh`
- 输出目录非空时会自动加 `_1`、`_2` 后缀

### 1. 默认训练（a800，8 卡）

In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.4
source .venv/bin/activate

export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7
export NPROC_PER_NODE=8
export NUM_EPOCHS=6
export MAX_LENGTH=4096
export NUM_DRAFT_LAYERS=5
export BLOCK_SIZE=16
export NUM_ANCHORS=512
export PIVOT_FUSE_MODE=linear_fuse
export NUM_MIDDLE_LAYERS_N=5
export LOSS_DECAY_GAMMA=7
export REPORT_TO=wandb

bash scripts/run_training_flashmtp.sh --dt h100

### 2. Hard Anchor Mining（低 prefix 接受长度 anchor 多练）

对同一条样本里 **EMA prefix 接受长度 ≤ threshold** 的 anchor 提高采样概率。

- `HARD_ANCHOR_MODE=weighted`：提高采样权重（默认）
- `HARD_ANCHOR_MODE=mixture`：预留 `HARD_ANCHOR_RATIO` 比例的 anchor slot 给 hard bank

In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.4
source .venv/bin/activate

export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7
export NPROC_PER_NODE=8
export NUM_EPOCHS=6
export MAX_LENGTH=4096
export NUM_DRAFT_LAYERS=5
export BLOCK_SIZE=16
export PIVOT_FUSE_MODE=linear_fuse
export NUM_MIDDLE_LAYERS_N=5
export LOSS_DECAY_GAMMA=7

# hard anchor mining
export HARD_ANCHOR_MINING=true
export HARD_ANCHOR_MODE=weighted # weighted 模式 # 或 mixture
export HARD_ANCHOR_THRESHOLD=2.5 # weighted 模式权重倍数
export HARD_ANCHOR_BOOST=8.0 # weighted 模式权重上限
export HARD_ANCHOR_MIN_VISITS=2  # 至少见过 2 次才认定为 hard

bash scripts/run_training_flashmtp.sh --dt h100

### 3. qz 集群（WandB offline）

In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.4
source .venv/bin/activate

export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7
export NPROC_PER_NODE=8
export DATA_NUM_SAMPLES=40000
export ENABLE_THINKING=off
export NUM_EPOCHS=6
export MAX_LENGTH=4096
export NUM_DRAFT_LAYERS=5
export BLOCK_SIZE=16
export PIVOT_FUSE_MODE=prefix_condition
export NUM_MIDDLE_LAYERS_N=5
export LOCAL_POSITION=true
export LOSS_DECAY_GAMMA=7
export WANDB_MODE=offline

bash scripts/run_training_flashmtp.sh --dt qz

### 4. 恢复训练

In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.4
source .venv/bin/activate

export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7
export NPROC_PER_NODE=8

# 方式 A：从 output_dir 自动找最新 checkpoint
export RESUME=true
export OUTPUT_DIR=./cache/models/flashmtp_a800_linear_fuse_fuse5_nemotron_40000_think_on_nlayers5_maxlen4096_epochs6_tlmh0_lp0_tmc0_an0_w1mse0_ham0

# 方式 B：指定 checkpoint 目录
# export CKPT_DIR=./cache/models/flashmtp_a800_.../epoch_3_step_15000

bash scripts/run_training_flashmtp.sh --dt a800

### 5. Ablation 常用开关

| 环境变量 | 说明 |
|---|---|
| `PIVOT_FUSE_MODE` | `linear_fuse` / `attention_fuse` / `prefix_condition` |
| `NUM_MIDDLE_LAYERS_N` | 中间 teacher 层数 N（总选取 = 2+N） |
| `TRAIN_LM_HEAD=true` | 单独训练 draft lm_head |
| `LOCAL_POSITION=true` | draft 块内 position 1..block_size |
| `LOSS_TEACHER_MATCH_CAP=true` | p_draft > p_teacher 时压 CE 权重 |
| `ADD_NOISE=true` | target hidden 加 U(-ratio, ratio) 噪声 |
| `W1_MSE=0.1` | 首个预测 token hidden MSE 权重 |
| `HARD_ANCHOR_MINING=true` | 低 prefix anchor oversample |

In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.4
source .venv/bin/activate

export CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7
export NPROC_PER_NODE=8
export NUM_EPOCHS=6
export MAX_LENGTH=4096
export NUM_DRAFT_LAYERS=5
export BLOCK_SIZE=16
export LOSS_DECAY_GAMMA=7

# ablation 示例：prefix_condition + local_position + hard anchor mixture
export PIVOT_FUSE_MODE=prefix_condition
export NUM_MIDDLE_LAYERS_N=5
export LOCAL_POSITION=true
export TRAIN_LM_HEAD=false
export HARD_ANCHOR_MINING=true
export HARD_ANCHOR_MODE=mixture
export HARD_ANCHOR_RATIO=0.3
export HARD_ANCHOR_THRESHOLD=2.0

bash scripts/run_training_flashmtp.sh --dt a800

### 6. 单卡调试（快速 smoke test）

In [ ]:
cd /data/wanghanzhen/Projects/MTP/NIPS26/FlashMTP_v1.4
source .venv/bin/activate

export CUDA_VISIBLE_DEVICES=0
export NPROC_PER_NODE=1
export NUM_EPOCHS=1
export MAX_LENGTH=2048
export NUM_ANCHORS=64
export NUM_DRAFT_LAYERS=1
export BLOCK_SIZE=16
export LOG_INTERVAL=1
export SAVE_INTERVAL=999999
export REPORT_TO=none

bash scripts/run_training_flashmtp.sh --dt a800